In [ ]:
pip install -r requirements.txt

## Polarisation Assessment

In [18]:
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.preprocessing import StandardScaler
import os

# Load the encoded dataset
df_encoded = pd.read_csv('../data/processed/wvs_encoded.csv')

# From the processed folder load and read the dataset for the four waves
df_wave4 = pd.read_csv('../data/processed/wvs_wave4.csv')
df_wave5 = pd.read_csv('../data/processed/wvs_wave5.csv')
df_wave6 = pd.read_csv('../data/processed/wvs_wave6.csv')
df_wave7 = pd.read_csv('../data/processed/wvs_wave7.csv')

In [20]:
from sklearn.decomposition import LatentDirichletAllocation
import pandas as pd
import json

# Optional: Load variable name mappings if you want labeled output
with open("variable_dict.json", "r") as f:
    variable_dict = json.load(f)

# Set fixed number of topics per wave (K=4 across all)
fixed_k = 4
waves = [4, 5, 6, 7]

# Map each wave to its dataframe
wave_data = {
    4: df_wave4,
    5: df_wave5,
    6: df_wave6,
    7: df_wave7
}

# Store results
k4_top_issues_by_wave = {}

In [23]:
# Process each wave with K=4
for wave_num in waves:
    print(f"\nProcessing Wave {wave_num} with K={fixed_k}")
    
    df = wave_data[wave_num]
    
    # Drop metadata columns
    lda_data = df.drop(columns=["country", "year"])
    
    # Fit LDA model
    lda_model = LatentDirichletAllocation(
          n_components=fixed_k,
          doc_topic_prior=0.25,
          topic_word_prior=0.1,
          learning_method='online',
          learning_decay=0.7,
          learning_offset=10.0,
          max_iter=20,
          batch_size=1000,
          evaluate_every=-1,
          mean_change_tol=0.001,
          max_doc_update_iter=100,
          n_jobs=-1,  # Uses all cores for this model
          random_state=25
    )
    lda_model.fit(lda_data)
    
    # Extract topic-word matrix and normalize
    topic_words = pd.DataFrame(lda_model.components_, columns=lda_data.columns)
    topic_words = topic_words.div(topic_words.sum(axis=1), axis=0)
    topic_words = topic_words.T
    topic_words.columns = [f"Ideology_{i+1}" for i in range(fixed_k)]
    
    # Get top 10 issues per ideology type
    top_issues = topic_words.apply(lambda x: x.nlargest(10).index.tolist(), axis=0)
    
    # Optional: label issues using variable_dict
    labeled_top_issues = top_issues.applymap(lambda code: variable_dict.get(code, code))
    
    # Save both labeled and raw codes
    labeled_top_issues.to_csv(f"wave{wave_num}_top_issues_labeled_k4.csv", index=False)
    top_issues.to_csv(f"wave{wave_num}_top_issues_k4.csv", index=False)

    # Store for further use
    k4_top_issues_by_wave[wave_num] = labeled_top_issues
    
    print(f"Saved top 10 issues for each ideology in Wave {wave_num}")


Processing Wave 4 with K=4


/var/folders/vk/c6csf7ws6pj9wy406twfjkfw0000gn/T/ipykernel_7293/982609394.py:38: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  labeled_top_issues = top_issues.applymap(lambda code: variable_dict.get(code, code))


Saved top 10 issues for each ideology in Wave 4

Processing Wave 5 with K=4


/var/folders/vk/c6csf7ws6pj9wy406twfjkfw0000gn/T/ipykernel_7293/982609394.py:38: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  labeled_top_issues = top_issues.applymap(lambda code: variable_dict.get(code, code))


Saved top 10 issues for each ideology in Wave 5

Processing Wave 6 with K=4


/var/folders/vk/c6csf7ws6pj9wy406twfjkfw0000gn/T/ipykernel_7293/982609394.py:38: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  labeled_top_issues = top_issues.applymap(lambda code: variable_dict.get(code, code))


Saved top 10 issues for each ideology in Wave 6

Processing Wave 7 with K=4
Saved top 10 issues for each ideology in Wave 7


/var/folders/vk/c6csf7ws6pj9wy406twfjkfw0000gn/T/ipykernel_7293/982609394.py:38: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  labeled_top_issues = top_issues.applymap(lambda code: variable_dict.get(code, code))


In [24]:
import json

# Load variable dictionary
with open("variable_dict.json", "r") as f:
    variable_dict = json.load(f)

# Extend it for support/oppose columns
extended_dict = {}
for var, label in variable_dict.items():
    extended_dict[f"{var}_support"] = f"{label} (Support)"
    extended_dict[f"{var}_oppose"] = f"{label} (Oppose)"

# Apply to top issues
labeled_top_issues = top_issues.apply(lambda col: col.map(extended_dict))